In [2]:
from IPython.core.display import display, HTML

display(HTML("""
<style>

/* ===== Layout ===== */
.container {
    width: 96% !important;
    max-width: 1600px !important;
    margin-left: auto !important;
    margin-right: auto !important;
}

/* ===== Base Styling ===== */
body {
    background-color: #f4f5f7;
    font-family: "Inter", "Segoe UI", Roboto, sans-serif;
    font-size: 15px;
    color: #2d2d2d;
    line-height: 1.55;
}

/* ===== Code Cells ===== */
div.input_area {
    background: #ffffff !important;
    border: 1px solid #d9d9d9 !important;
    border-radius: 8px !important;
    padding: 10px !important;
}

.CodeMirror {
    font-family: "Fira Code", "Source Code Pro", monospace;
    font-size: 13.5px;
}

/* ===== OUTPUT: Clean, simple, professional ===== */
div.output_wrapper, div.output {
    background: #ffffff !important;
    border: 1px solid #e1e1e1 !important;
    border-radius: 6px !important;
    padding: 10px 14px !important;
    margin-top: 8px !important;
}

/* ===== Markdown / Text Cells ===== */
.text_cell_render {
    background: #ffffff;
    border-radius: 8px;
    padding: 18px;
    margin-bottom: 14px;
    border: 1px solid #e2e2e2;
}

/* ===== Professional Headings ===== */
h1, h2, h3, h4 {
    font-family: "Inter", sans-serif;
    font-weight: 600;
    color: #1f2a44;
}
h1 { font-size: 1.85em; border-bottom: 1px solid #d9dee7; padding-bottom: 6px; }
h2 { font-size: 1.55em; }
h3 { font-size: 1.28em; }

/* ===== Tables ===== */
table {
    border-collapse: collapse;
    width: 100%;
}
th, td {
    border: 1px solid #d7d7d7;
    padding: 8px 12px;
}
th {
    background: #eef1f5;
    font-weight: 600;
}

/* ===== Scrollbar (Minimal) ===== */
::-webkit-scrollbar { width: 7px; }
::-webkit-scrollbar-thumb {
    background: #b8c1cb;
    border-radius: 10px;
}

/* ===== Links ===== */
a {
    color: #004b9c;
}

/* ===== Cell spacing ===== */
.cell {
    margin-top: 16px;
    margin-bottom: 16px;
}

</style>
"""))


/tmp/ipykernel_1934061/2771030678.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [3]:
import  os
import fitz  # PyMuPDF
# from langgraph_utils.Info_extractor import InfoExtractorAgent
# from langgraph_utils.chat_history import SnowflakeChatMessageHistory
import json
# from langgraph_utils.file_parser import FileParser
# from langgraph_utils.digitization import main_handler
import uuid
# from langgraph_utils import creds
# from langgraph_utils.variables import PROJECT_NAME, SECRET_NAME, TOKEN_KEY
from utils.connection import get_dataiku_client_and_project
import logging
import tempfile
import re
import pymupdf4llm
from dataikuapi import DSSClient
from dataikuapi.dss.project import DSSProject
import pandas as pd
import asyncio

from soa_extraction.opensearch_utils import OpensearchUtil, create_embeddings_new
import nest_asyncio



# import from GLOBAL SHARED CODE
from utils import connection 
# imports from library 
# from utilities.creds import RD_PROJECT_NAME
# from variables import SECRET_NAME , TOKEN_KEY
# from utilities.logging_config import logging
from IPython.core.display import display, HTML
from uuid import uuid4

/tmp/ipykernel_1934061/646784850.py:32: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [4]:
import dataikuapi
DATAIKU_HOST = "http://10.45.152.66:10000"
API_SECRET_KEY = "dkuaps-b3EsRXVjU3w4y7nd4KEwibEr04CjFPZr"          
PROJECT_NAME = "ECSGENERATION"   

# client, proj = get_dataiku_client_and_project(PROJECT_NAME, SECRET_NAME, TOKEN_KEY)
client = dataikuapi.DSSClient(DATAIKU_HOST, API_SECRET_KEY)
proj = client.get_project(PROJECT_NAME)

In [5]:
# self.proj = proj
# self.client = client
s3_folder_dataset_id = proj.get_variables()['local'].get('ecs_excel') # change file upload 
input_folder = proj.get_managed_folder(s3_folder_dataset_id)
files = input_folder.list_contents()["items"]
toc_page_limit = 20
config = proj.get_variables()["local"]

In [6]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- DATAIKU FOLDER READ ---
s3_folder_dataset_id = proj.get_variables()['local'].get('ecs_excel')
input_folder = proj.get_managed_folder(s3_folder_dataset_id)
files = input_folder.list_contents()["items"]

# Extract file names
file_names = [f["path"] for f in files]

# --- STANDARD COLUMNS ---
standard_columns = [
    "validation id", "OGCMS Version", "data collection domain name",
    "data collection domain CRF", "data collection variable text",
    "variable name", "Validation Logic", "Reasoning",
    "Action", "action details"
]

# --- GLOBAL VARIABLES ---
current_file_index = 0
all_final_dfs = []

# --- OUTPUT WIDGET ---
output = widgets.Output()

# --- WIDGETS ---
file_label = widgets.Label()
sheet_dropdown = widgets.Dropdown(description="Select Sheet:")
select_sheet_button = widgets.Button(description="Load Sheet", button_style='info')
generate_button = widgets.Button(description="Generate & Next File", button_style='warning')
mapping_widgets = {}
available_cols = []

def load_file(file_index):
    """Load a file from file_names by index"""
    with output:
        clear_output()
        global xls, selected_file
        if file_index >= len(file_names):
            print("✅ All files processed!")
            if all_final_dfs:
                merged_df = pd.concat(all_final_dfs, ignore_index=True)
                merged_df.to_excel("merged_standardized_output.xlsx", index=False)
                print("💾 Merged output saved as: merged_standardized_output.xlsx")
            return

        selected_file = file_names[file_index]
        # --- Display current file being processed ---
        file_label.value = f"Processing file {file_index+1} of {len(file_names)}: {selected_file}"
          # Show file path first

        try:
            print(file_label.value)
            with input_folder.get_file(selected_file) as stream:
                file_bytes = stream.raw.data
            xls = pd.ExcelFile(file_bytes)
            sheet_dropdown.options = xls.sheet_names

            # --- Display sheet dropdown and button after showing file ---
            display(sheet_dropdown, select_sheet_button)

        except Exception as e:
            print("❌ Error loading file:", e)

def load_sheet(b):
    """Load selected sheet and show mapping UI"""
    with output:
        clear_output()
        global df, available_cols, mapping_widgets
        selected_file = file_names[current_file_index]

        with input_folder.get_file(selected_file) as stream:
            file_bytes = stream.raw.data
        df = pd.read_excel(file_bytes, sheet_name=sheet_dropdown.value)
        display(file_label)
        display(df.head())

        available_cols = list(df.columns)
        mapping_widgets.clear()

        print("\n### Map Columns ###")
        for col in standard_columns:
            dropdown = widgets.Dropdown(
                options=[None] + available_cols,
                description=col + ":",
                layout=widgets.Layout(width='700px'),
                style={'description_width': '250px'}
            )
            mapping_widgets[col] = dropdown
            display(dropdown)

        display(generate_button)

def generate_output(b):
    """Generate standardized output for current file and move to next"""
    global current_file_index, all_final_dfs
    with output:
        clear_output()
        final_df = pd.DataFrame()

        for std_col, dropdown in mapping_widgets.items():
            selected_col = dropdown.value
            final_df[std_col] = df[selected_col] if selected_col else None

        # Save current file output
        output_filename = f"standardized_{current_file_index+1}.xlsx"
        final_df.to_excel(output_filename, index=False)
        print(f"✅ Standardized File Saved as: {output_filename}")

        all_final_dfs.append(final_df)

        # Move to next file
        current_file_index += 1
        load_file(current_file_index)

# --- BUTTON CALLBACKS ---
select_sheet_button.on_click(load_sheet)
generate_button.on_click(generate_output)

# --- INITIAL DISPLAY ---
load_file(current_file_index)
display(output)


Processing file 1 of 11: /Histoical/384-201-00004_Database Check Guide_04MAR2025.xlsx


/tmp/ipykernel_1934061/1678492737.py:58: FutureWarning: Passing bytes to 'read_excel' is deprecated and will be removed in a future version. To read from a byte string, wrap it in a `BytesIO` object.
  xls = pd.ExcelFile(file_bytes)


Dropdown(description='Select Sheet:', options=('Cover Page', 'Edit Check', 'Visibility', 'Methods', 'Test Case…

Button(button_style='info', description='Load Sheet', style=ButtonStyle())

Output()

In [55]:
for i in all_final_dfs:
    print(i.shape)

(359, 10)
(637, 10)
(751, 10)
(496, 10)


In [11]:
class readExcel:
    def __init__(self,client,proj):
        self.proj = proj
        self.client = client
        self.s3_folder_dataset_id = proj.get_variables()['local'].get('ecs_excel') # change file upload 
        self.input_folder = proj.get_managed_folder(self.s3_folder_dataset_id)
        self.files = self.input_folder.list_contents()["items"]
        self.toc_page_limit = 20
        self.config = proj.get_variables()["local"]
        print(self.files)
        
    def standard_ecs_excel_read(self,path= ''):
        """
        This Class Function is responsible for reading the Standard Excel Files
        """
        try:
            result = []

            for file in self.files:
                path = file['path']
                print(path)
                parts = path.strip('/').split('/')
                if file_path == '':
                    if "Standard" in path:

                        result.append({

                            "path": path,


                        })
                else:
                      path = file_path  
                    
                
            df_list = []
            for file in result:
                if file_path == '':
                    with self.input_folder.get_file(file['path']) as stream:
                        file_bytes = stream.raw.data
                    #     df = pd.read_excel(file_bytes, sheet_name='SV Domain Validations')  
                        sheet_names = pd.ExcelFile(file_bytes).sheet_names
                        
                        
                combined_df = pd.DataFrame()
                for sheet in sheet_names:
                    if 'Rave' in sheet:
                        print('do not read the excel')
                        continue
                    with self.input_folder.get_file(file['path']) as stream:
                        file_bytes = stream.raw.data
                        dict_df = pd.read_excel(file_bytes, sheet_name=sheet)
                #     print(type(dict_df))    
                #     temp_df = pd.DataFrame([dict_df])
                    temp_df = dict_df
                    combined_df = pd.concat([combined_df,temp_df],ignore_index=True)
                df_list.append(combined_df)
                
            return df_list
        except:
            import traceback
            t = traceback.format_exc()
            raise f"error caused due to {t}"

In [6]:
obj = readExcel(client,proj)

[{'path': '/Histoical/384-201-00004_Database Check Guide_04MAR2025.xlsx', 'size': 154645, 'lastModified': 1758714856000}, {'path': '/Histoical/MAC186_X11-201-00001_Data Validation Plan_V6.0 15Aug2025.xlsx', 'size': 168923, 'lastModified': 1758714856000}, {'path': '/Histoical/Otsuka 384-201-00002_Data Validation Specification_V4.0_29Apr2025.xlsx', 'size': 1104146, 'lastModified': 1758714838000}, {'path': '/Standard/Copy of Otsuka Standard Edit Check Specifications (1).xlsx', 'size': 131156, 'lastModified': 1757487453000}]


# /Histoical/Otsuka 384-201-00002_Data Validation Specification_V4.0_29Apr2025.xlsx

In [7]:
with input_folder.get_file("/Histoical/Otsuka 384-201-00002_Data Validation Specification_V4.0_29Apr2025.xlsx") as stream:
    file_bytes = stream.raw.data
    dict_df = pd.read_excel(file_bytes, sheet_name="Edit Checks")

/tmp/ipykernel_3463641/2704211350.py:3: FutureWarning: Passing bytes to 'read_excel' is deprecated and will be removed in a future version. To read from a byte string, wrap it in a `BytesIO` object.
  dict_df = pd.read_excel(file_bytes, sheet_name="Edit Checks")


In [14]:
dict_df

,No.,Form,Validation Name,CF?,Library Name,Needs Retesting,User Requirement,Logic (Design Spec),Action,Message,...,Testing Comments.1,UAT Results\n(specifics provided by LCDSci).1,CRF Version # (must be same as or greater than version used for Unit Testing by CP).2,Clean Subject(s) #.2,Dirty Subject(s) #.2,UAT Results\n(Pass/Fail).2,Tester Initials.2,Testing Date \n(DD-Mmm-YYYY).2,Testing Comments.2,UAT Results\n(specifics provided by LCDSci).2
0,1,Abnormal Involuntary Movement Scale (AIMS),RS_AIMS004,No,NaN,No,RSDTC & RSTIM is not 8 hours after EC.ECSTTIM ...,NaN,Open Query,Assessment Date & Time is not within 8 hours p...,...,17-Mar-2025 KH: Check worked as expected on th...,Publish check 2,NaN,NaN,NaN,NaN,NaN,NaT,NaN,Publish check 2
1,2,Barnes Akathisia Rating Scale (BARS),RS_BARS004,No,NaN,No,RSDTC & RSTIM is not within 8 hours after EC.E...,NaN,Open Query,Assessment Date & Time is not within 8 hours p...,...,NaN,Publish check 2,1902-03-14 00:00:00,996-S50062,996-S50062,Pass,KH,2025-03-18,18-Mar-2025 KH: Tested on the following visits...,Publish check 2
2,3,Calgary Depression Scale for Schizophrenia (CDSS),QS_CDSS003,No,NaN,No,QS_CDSS.QSDTC AND QS_CDSS.QSTIM is not within ...,NaN,Open Query,Assessment Date & Time is not within 8 hours p...,...,17-Mar-2025 KH: Check worked as expected on th...,Publish check 2,NaN,NaN,NaN,NaN,NaN,NaT,NaN,Publish check 2
3,4,Central Laboratory,LB_CNTRL004,No,NaN,No,[within visit] Flag if LB_CNTRL.LBDAT & LB_CNT...,NaN,Open Query,The Collection Date & Time is not prior to the...,...,17-Mar-2025 KH: Tested on the following visits...,Publish check 2,NaN,NaN,NaN,NaN,NaN,NaT,NaN,Publish check 2
4,5,Clinical Global Impression - Severity of Illne...,QS_CGIS004,No,NaN,No,QSDTC & QSTIM is not within 8 hours after EC.E...,NaN,Open Query,Assessment Date & Time is not within 8 hours p...,...,17-Mar-2025 KH: Check worked as expected on th...,Publish check 2,NaN,NaN,NaN,NaN,NaN,NaT,NaN,Publish check 2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
746,747,Structured Interview Guide for the Hamilton An...,DYN_RS_SIGHA_RSYN_SET_VIS,NaN,NaN,No,"If RSYN IsEqualTo Y then... RSYN IsPresent, an...",NaN,Set Datapoint Visible,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
747,748,Structured Interview Guide for the Montgomery ...,DYN_RS_SIGMA_RSYN_SET_VIS,NaN,NaN,No,"If RSYN IsEqualTo Y then... RSYN IsPresent, an...",NaN,Set Datapoint Visible,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
748,749,Structured Interview Guide for the Montgomery ...,DYN_RS_SIGMA_SLE_RSYN_SET_VIS,NaN,NaN,No,"If RSYN IsEqualTo Y then... RSYN IsPresent, an...",NaN,Set Datapoint Visible,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
749,750,Structured Interview Manual for Young Mania Ra...,DYN_RS_YMRS_RSYN_SET_VIS,NaN,NaN,No,"If RSYN IsEqualTo Y then... RSYN IsPresent, an...",NaN,Set Datapoint Visible,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN


In [21]:
 for u in dict_df.columns:
        print(u)

No.
Form
Validation Name
CF?
Library Name
Needs Retesting
User Requirement
Logic (Design Spec)
Action
Message
Current UAT Status
CRF Version # (must be same as or greater than version used for Unit Testing by CP)
Clean Subject(s) #
Dirty Subject(s) #
UAT Results
(Pass/Fail)
Tester Initials
Testing Date 
(DD-Mmm-YYYY)
Testing Comments
UAT Results
(specifics provided by LCDSci)
CRF Version # (must be same as or greater than version used for Unit Testing by CP).1
Clean Subject(s) #.1
Dirty Subject(s) #.1
UAT Results
(Pass/Fail).1
Tester Initials.1
Testing Date 
(DD-Mmm-YYYY).1
Testing Comments.1
UAT Results
(specifics provided by LCDSci).1
CRF Version # (must be same as or greater than version used for Unit Testing by CP).2
Clean Subject(s) #.2
Dirty Subject(s) #.2
UAT Results
(Pass/Fail).2
Tester Initials.2
Testing Date 
(DD-Mmm-YYYY).2
Testing Comments.2
UAT Results
(specifics provided by LCDSci).2


In [8]:
final_df = dict_df[['Validation Name','Form','User Requirement','Message','Action']]

In [9]:
final_df.head()

,Validation Name,Form,User Requirement,Message,Action
0,RS_AIMS004,Abnormal Involuntary Movement Scale (AIMS),RSDTC & RSTIM is not 8 hours after EC.ECSTTIM ...,Assessment Date & Time is not within 8 hours p...,Open Query
1,RS_BARS004,Barnes Akathisia Rating Scale (BARS),RSDTC & RSTIM is not within 8 hours after EC.E...,Assessment Date & Time is not within 8 hours p...,Open Query
2,QS_CDSS003,Calgary Depression Scale for Schizophrenia (CDSS),QS_CDSS.QSDTC AND QS_CDSS.QSTIM is not within ...,Assessment Date & Time is not within 8 hours p...,Open Query
3,LB_CNTRL004,Central Laboratory,[within visit] Flag if LB_CNTRL.LBDAT & LB_CNT...,The Collection Date & Time is not prior to the...,Open Query
4,QS_CGIS004,Clinical Global Impression - Severity of Illne...,QSDTC & QSTIM is not within 8 hours after EC.E...,Assessment Date & Time is not within 8 hours p...,Open Query


In [10]:
name_uuid_map = {}
name_ids = []
row_ids = []
import uuid
for index, row in final_df.iterrows():
    name = row["Form"]
    if name not in name_uuid_map:
        name_uuid_map[name] = str(uuid.uuid4())
    name_ids.append(name_uuid_map[name])
    
    # Assign new UUID for each row
    row_ids.append(str(uuid.uuid4()))

final_df['form_name_id'] = name_ids
final_df['row_id'] = row_ids
# ecs_list[0]

/tmp/ipykernel_3463641/4291044367.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['form_name_id'] = name_ids
/tmp/ipykernel_3463641/4291044367.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['row_id'] = row_ids


In [29]:
final_df

,Validation Name,Form,User Requirement,Message,Action,form_name_id,row_id
0,RS_AIMS004,Abnormal Involuntary Movement Scale (AIMS),RSDTC & RSTIM is not 8 hours after EC.ECSTTIM ...,Assessment Date & Time is not within 8 hours p...,Open Query,2fa7e859-67d5-4e8f-94db-014abd5af37c,3f8c23d7-ff01-4bda-9451-18049eaab14f
1,RS_BARS004,Barnes Akathisia Rating Scale (BARS),RSDTC & RSTIM is not within 8 hours after EC.E...,Assessment Date & Time is not within 8 hours p...,Open Query,05374a23-1e4a-42f2-9fb6-fc078fd865f2,8e8b76c3-b86b-4274-96bc-43cbe62f442b
2,QS_CDSS003,Calgary Depression Scale for Schizophrenia (CDSS),QS_CDSS.QSDTC AND QS_CDSS.QSTIM is not within ...,Assessment Date & Time is not within 8 hours p...,Open Query,53b27c23-d60f-48c0-867a-6d3f6f794d85,ae6103ec-2344-4c78-9828-bcc566e1b506
3,LB_CNTRL004,Central Laboratory,[within visit] Flag if LB_CNTRL.LBDAT & LB_CNT...,The Collection Date & Time is not prior to the...,Open Query,53abbd8a-4e77-4b1f-ad53-e22174516e1b,ca4c7f28-e965-4058-9012-728a614b13a9
4,QS_CGIS004,Clinical Global Impression - Severity of Illne...,QSDTC & QSTIM is not within 8 hours after EC.E...,Assessment Date & Time is not within 8 hours p...,Open Query,b54503c6-a25f-44d2-bf8b-7ad22b000b67,963390b4-26e9-4022-ae7c-9bf56f39de3a
...,...,...,...,...,...,...,...
746,DYN_RS_SIGHA_RSYN_SET_VIS,Structured Interview Guide for the Hamilton An...,"If RSYN IsEqualTo Y then... RSYN IsPresent, an...",NaN,Set Datapoint Visible,28dd4df8-4437-40df-9cdd-a114cf29a21a,c478df78-93b9-417b-bd58-0d40e3a923eb
747,DYN_RS_SIGMA_RSYN_SET_VIS,Structured Interview Guide for the Montgomery ...,"If RSYN IsEqualTo Y then... RSYN IsPresent, an...",NaN,Set Datapoint Visible,d8d882d1-ffc0-441b-8fa2-4659b7e381a6,7d1893ea-e7d9-44cb-bc3f-4debbe54c13f
748,DYN_RS_SIGMA_SLE_RSYN_SET_VIS,Structured Interview Guide for the Montgomery ...,"If RSYN IsEqualTo Y then... RSYN IsPresent, an...",NaN,Set Datapoint Visible,500648d7-f548-4e14-a209-7821de6a5298,0e068424-1c5c-40e9-b3f0-42a34ec724b2
749,DYN_RS_YMRS_RSYN_SET_VIS,Structured Interview Manual for Young Mania Ra...,"If RSYN IsEqualTo Y then... RSYN IsPresent, an...",NaN,Set Datapoint Visible,533dbc05-1f4c-4274-8291-73c192105d18,4ec3d5a0-13c5-46f4-9484-0deaa6a5ced5


In [11]:
form_name_list = []
field_value_list = []
mix_list = []
for index, row in final_df.iterrows():
    form_name_list.append(row['Form'])
# final__list_2 = form_name_list[:100]   

In [17]:
nest_asyncio.apply()

form_embeddings =  asyncio.run(create_embeddings_new(proj, form_name_list, proj.get_variables()['local'].get('default_embeddings_model_id')))

In [18]:
form_out =  []
form_out.append(form_embeddings['response'])

In [31]:
len(form_embeddings['response'])
final_df = final_df.fillna("")

In [32]:
generic_mapping = []
for (index,row) , form_emb in zip(final_df.iterrows()
                                                    ,form_embeddings['response'],
                                                    ):
    generic_mapping.append(
        {
            "ecs_id": row['row_id'],
            "form_id" : row['form_name_id'],
            "validation_id": row['Validation Name'],
            "form_name": row['Form'],
            
            "validation_logic":row['User Requirement'],
         
            "action": row['Action'],
            "action_details": row['Message'],
            "source": "Historic",
            "path":"/Histoical/Otsuka 384-201-00002_Data Validation Specification_V4.0_29Apr2025.xlsx",
            "form_name_vector":form_emb,
            
            
            
        }
    )

In [33]:
len(generic_mapping)

751

In [37]:
import json
from opensearchpy.helpers import bulk
from opensearchpy import OpenSearch
opensearch_client = OpensearchUtil(client, proj)

def bulk_insert( index_name, documents):
    project = proj
        
    # Retrieve credentials from Opensearch connection 
    project_configs = project.get_variables()["local"]
    opensearch_connection = project_configs["opensearch_connection"]



    conn_info = client.get_connection(opensearch_connection).get_info()

    opensearchclient = OpenSearch(
            hosts=[{"host": conn_info["params"]["host"], "port": conn_info["params"]["port"]}],
            http_auth = (conn_info["params"]["username"], conn_info["params"]["password"]),
            use_ssl=conn_info["params"]["ssl"],
            verify_certs=False
    )
    

    docs_to_store = []
    for document in documents:

        action = {
            "_op_type": "index",  # Operation type (index = insert)
            "_index": index_name,  # Index name
            "_id": document['ecs_id'],  # Use the unique document ID
            "_source": document  # Document body
        }
        docs_to_store.append(action)

    success, failed = bulk(opensearchclient, docs_to_store)
    print(f"Successfullly indexed {success} documents.")
    print(f"Failed to index {failed} documents.")
    if failed:
        raise Exception("Document indexing encountered an unknown error. ")


# index_name = proj.get_variables()['local'].get('ecs_opensearch')
index_name = "${projectKey}_ecs_index"
dataiku_project_var = "${projectKey}"
if dataiku_project_var in index_name:
    index_name = index_name.replace(dataiku_project_var, opensearch_client.project.project_key).lower()
    print("index_name",index_name)
bulk_insert(index_name, generic_mapping)

{'type': 'ElasticSearch', 'params': {'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'username': 'genai-admin', 'password': 'qPk7Jf5vcyXDS!**332gXTvSfmcauvr9', 'port': 443, 'ssl': True, 'trustAnySSLCertificate': True, 'dialect': 'ES_7', 'dkuProperties': [], 'namingRule': {'indexNameDatasetNamePrefix': '${projectKey}_'}, 'authType': 'PASSWORD', 'oauth': {'refreshTokenRotation': False}, 'aws': {'service': 'OPENSEARCH_SERVERLESS', 'credentialsMode': 'KEYPAIR', 'customAWSCredentialsProviderParams': []}}, 'credentialsMode': 'GLOBAL', 'proxySettingsAsString': ''}
opensearch <OpenSearch([{'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'port': 443}])>
index_name ecsgeneration_ecs_index


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/opensearchpy/connection/http_urllib3.py:214: UserWarning: Connecting to https://aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com:443 using SSL with verify_certs=False is insecure.
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


Successfullly indexed 751 documents.
Failed to index [] documents.


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


,Validation Name,Form,User Requirement,Message,Action,form_name_id,row_id
0,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...
746,False,False,False,False,False,False,False
747,False,False,False,False,False,False,False
748,False,False,False,False,False,False,False
749,False,False,False,False,False,False,False


# Histoical/384-201-00004_Database Check Guide_04MAR2025.xlsx

In [41]:
with input_folder.get_file("/Histoical/384-201-00004_Database Check Guide_04MAR2025.xlsx") as stream:
    file_bytes = stream.raw.data
    dict_df = pd.read_excel(file_bytes, sheet_name="Edit Check")

/tmp/ipykernel_491563/3001530861.py:3: FutureWarning: Passing bytes to 'read_excel' is deprecated and will be removed in a future version. To read from a byte string, wrap it in a `BytesIO` object.
  dict_df = pd.read_excel(file_bytes, sheet_name="Edit Check")


In [42]:
dict_df.columns

Index(['Study_Name', 'Item_Origin', 'Form_Name', 'Item_Group_Name',
       'Item_Name', 'Item_Prompt', 'Is_edit_check_soft', 'Edit_Check_name',
       'Formal_Expression_Context', 'Edit_Check_Description',
       'Edit_Check_Error_Message', 'Edit_Check_Value', 'Soft_Range',
       'Hard_Range'],
      dtype='object')

In [43]:
final_df = dict_df[['Form_Name','Item_Prompt','Edit_Check_name','Edit_Check_Description','Edit_Check_Value','Edit_Check_Error_Message']]

In [48]:
final_df[final_df['Form_Name'] == 'Demographics']


,Form_Name,Item_Prompt,Edit_Check_name,Edit_Check_Description,Edit_Check_Value,Edit_Check_Error_Message
54,Demographics,Was the procedure performed?,TEMPLATE_PROCEDURE_PERFORMED_LEADING_QUESTION,Value should = Y,Must Equal Y,"Procedure is expected to be performed, please ..."
55,Demographics,Age,NaN,NaN,,NaN
56,Demographics,Race - Unknown,TEMPLATE_DMRACE_UNKNOWN,Query if no races selected or if both 'Race-Un...,,NaN


In [10]:
final_df
final_df = final_df.fillna("")

In [17]:
final_df['Item_Prompt'] == ""

0      False
1      False
2      False
3      False
4      False
       ...  
354    False
355    False
356    False
357    False
358    False
Name: Item_Prompt, Length: 359, dtype: bool

In [19]:
form_name_list = []
field_value_list = []
mix_list = []
for index, row in final_df.iterrows():
    form_name_list.append(row['Form_Name'])
    if (row['Item_Prompt'] == "" ):
         field_value_list.append('-1')
    else:
        field_value_list.append(row['Item_Prompt'])
# final__list_2 = form_name_list[:100]   

In [20]:
nest_asyncio.apply()

form_embeddings =  asyncio.run(create_embeddings_new(proj, form_name_list, proj.get_variables()['local'].get('default_embeddings_model_id')))




In [21]:
fields_embeddings =  asyncio.run(create_embeddings_new(proj, field_value_list, proj.get_variables()['local'].get('default_embeddings_model_id')))


In [29]:
name_uuid_map = {}
name_ids = []
row_ids = []
import uuid
for index, row in final_df.iterrows():
    name = row["Form_Name"]
    if name not in name_uuid_map:
        name_uuid_map[name] = str(uuid.uuid4())
    name_ids.append(name_uuid_map[name])
    
    # Assign new UUID for each row
    row_ids.append(str(uuid.uuid4()))

final_df['form_name_id'] = name_ids
final_df['row_id'] = row_ids
# ecs_list[0]

In [30]:
generic_mapping = []
for (index,row) , form_emb,field_emb in zip(final_df.iterrows()
                                                    ,form_embeddings['response'],
                                                    fields_embeddings['response'],
                                                   ):
    generic_mapping.append(
        {
            "ecs_id": row['row_id'],
            "form_id" : row['form_name_id'],
            
            "form_name": row['Form_Name'],
           
            "form_field_value": row['Item_Prompt'],
            "variable_name":row['Edit_Check_name'],
            "validation_logic":row['Edit_Check_Description'],
            
            "action": row['Edit_Check_Value'],
            "action_details": row['Edit_Check_Error_Message'],
            "source": "Historic",
            "path":"/Histoical/384-201-00004_Database Check Guide_04MAR2025.xlsx",
            "form_name_vector":form_emb,
            "form_field_value_vector":field_emb
            
            
        }
    )

In [31]:
import json
from opensearchpy.helpers import bulk
from opensearchpy import OpenSearch
opensearch_client = OpensearchUtil(client, proj)

def bulk_insert( index_name, documents):
    project = proj
        
    # Retrieve credentials from Opensearch connection 
    project_configs = project.get_variables()["local"]
    opensearch_connection = project_configs["opensearch_connection"]



    conn_info = client.get_connection(opensearch_connection).get_info()

    opensearchclient = OpenSearch(
            hosts=[{"host": conn_info["params"]["host"], "port": conn_info["params"]["port"]}],
            http_auth = (conn_info["params"]["username"], conn_info["params"]["password"]),
            use_ssl=conn_info["params"]["ssl"],
            verify_certs=False
    )
    

    docs_to_store = []
    for document in documents:

        action = {
            "_op_type": "index",  # Operation type (index = insert)
            "_index": index_name,  # Index name
            "_id": document['ecs_id'],  # Use the unique document ID
            "_source": document  # Document body
        }
        docs_to_store.append(action)

    success, failed = bulk(opensearchclient, docs_to_store)
    print(f"Successfullly indexed {success} documents.")
    print(f"Failed to index {failed} documents.")
    if failed:
        raise Exception("Document indexing encountered an unknown error. ")


# index_name = proj.get_variables()['local'].get('ecs_opensearch')
index_name = "${projectKey}_ecs_index"
dataiku_project_var = "${projectKey}"
if dataiku_project_var in index_name:
    index_name = index_name.replace(dataiku_project_var, opensearch_client.project.project_key).lower()
    print("index_name",index_name)
bulk_insert(index_name, generic_mapping)

{'type': 'ElasticSearch', 'params': {'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'username': 'genai-admin', 'password': 'qPk7Jf5vcyXDS!**332gXTvSfmcauvr9', 'port': 443, 'ssl': True, 'trustAnySSLCertificate': True, 'dialect': 'ES_7', 'dkuProperties': [], 'namingRule': {'indexNameDatasetNamePrefix': '${projectKey}_'}, 'authType': 'PASSWORD', 'oauth': {'refreshTokenRotation': False}, 'aws': {'service': 'OPENSEARCH_SERVERLESS', 'credentialsMode': 'KEYPAIR', 'customAWSCredentialsProviderParams': []}}, 'credentialsMode': 'GLOBAL', 'proxySettingsAsString': ''}
opensearch <OpenSearch([{'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'port': 443}])>
index_name ecsgeneration_ecs_index


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/opensearchpy/connection/http_urllib3.py:214: UserWarning: Connecting to https://aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com:443 using SSL with verify_certs=False is insecure.
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


Successfullly indexed 359 documents.
Failed to index [] documents.


# '/Histoical/MAC186_X11-201-00001_Data Validation Plan_V6.0 15Aug2025.xlsx'

In [50]:
with input_folder.get_file("/Histoical/MAC186_X11-201-00001_Data Validation Plan_V6.0 15Aug2025.xlsx") as stream:
    file_bytes = stream.raw.data
    dict_df = pd.read_excel(file_bytes, sheet_name="MAC186 Custom Queries")

/tmp/ipykernel_491563/1867866337.py:3: FutureWarning: Passing bytes to 'read_excel' is deprecated and will be removed in a future version. To read from a byte string, wrap it in a `BytesIO` object.
  dict_df = pd.read_excel(file_bytes, sheet_name="MAC186 Custom Queries")


In [51]:
dict_df.columns

Index(['Check ID', 'Form Name', 'Form Export Name', 'Variable',
       'Visit form is present', 'Visit check applies to', 'Check logic',
       'Programming logic\n(this column should be hidden prior to sending to team review)',
       'Query text', 'Designation', 'Fire Query', 'Send Email',
       'Email Subject', 'Send To', 'Do not show system-generated description',
       'DVP version updated at', 'Update made (eg New / Updated / Deleted)'],
      dtype='object')

In [55]:
final_df = dict_df[['Check ID','Form Name','Form Export Name','Variable','Programming logic\n(this column should be hidden prior to sending to team review)','Designation','Query text']]

In [76]:
final_df = final_df.fillna("")

In [77]:
name_uuid_map = {}
name_ids = []
row_ids = []
import uuid
for index, row in final_df.iterrows():
    name = row["Form Name"]
    if name not in name_uuid_map:
        name_uuid_map[name] = str(uuid.uuid4())
    name_ids.append(name_uuid_map[name])
    
    # Assign new UUID for each row
    row_ids.append(str(uuid.uuid4()))

final_df['form_name_id'] = name_ids
final_df['row_id'] = row_ids


In [70]:
final_df.fillna("")

,Check ID,Form Name,Form Export Name,Variable,Programming logic\n(this column should be hidden prior to sending to team review),Designation,Query text,form_name_id,row_id
0,DS_IC_001,Informed Consent,DS_IC,DSYN_IC,DSYN_IC (Informed Consent Obtained) = No,Invalid data,Informed Consent Obtained is No. Consent must ...,47e7b2c2-e8e9-4a90-b974-954037f3b624,1bff323a-0126-489d-8d38-c41ab6979843
1,DS_IC_902,Informed Consent,DS_IC,DSICVER,DSICVER (Informed Consent Version) = Part 1 v4...,Invalid data,The Informed Consent Date is before the date o...,47e7b2c2-e8e9-4a90-b974-954037f3b624,2599c595-683a-436e-bebb-54b06c736743
2,DS_IC_903,Informed Consent,DS_IC,DSICVER,DSICVER (Informed Consent Version) = Part 2 v4...,Invalid data,The Informed Consent Date is before the date o...,47e7b2c2-e8e9-4a90-b974-954037f3b624,4aa11f7c-ef47-42a5-af98-3398193add32
3,DS_IC_904,Informed Consent,DS_IC,DSICVER,DSICVER (Informed Consent Version) = Part 1 v4...,Invalid data,The Informed Consent Version is for Part 1 but...,47e7b2c2-e8e9-4a90-b974-954037f3b624,1734df2c-97d8-4a39-a7a8-2232f917470b
4,DS_IC_905,Informed Consent,DS_IC,DSICVER,DSICVER (Informed Consent Version) = Part 2 v4...,Invalid data,The Informed Consent Version is for Part 2 but...,47e7b2c2-e8e9-4a90-b974-954037f3b624,b5598705-52e4-4ba0-b13b-fa555d9bb30d
...,...,...,...,...,...,...,...,...,...
632,UPK_902,Urine PK Sampling,UPK,UPKSTDAT,At same visit:\nUPKSTDAT (Date of the first vo...,Out of range,8-16 hours: Date of the first void in the coll...,68c2eb98-24dc-46c1-9b9d-5ac3d016ea45,0bc1317e-8a89-4349-b6f5-983b438939b0
633,UPK_903,Urine PK Sampling,UPK,UPKSTDAT,At same visit:\nUPKSTDAT (Date of the first vo...,Out of range,16-24 hours: Date of the first void in the col...,68c2eb98-24dc-46c1-9b9d-5ac3d016ea45,150a9f19-3549-4930-91f3-b1c05903e785
634,UPK_905,Urine PK Sampling,UPK,UPKENDAT,At same visit:\nUPKENDAT (Date of the last voi...,Out of range,0-8 hours: Date of the last void in the collec...,68c2eb98-24dc-46c1-9b9d-5ac3d016ea45,ca48b2a8-e405-4b49-89e4-0374cb4f2615
635,UPK_906,Urine PK Sampling,UPK,UPKENDAT,At same visit:\nUPKENDAT (Date of the last voi...,Out of range,8-16 hours: Date of the last void in the colle...,68c2eb98-24dc-46c1-9b9d-5ac3d016ea45,7a911c09-dd4a-40d6-8a33-d406b0b18696


In [78]:
form_name_list = []
field_value_list = []
mix_list = []
for index, row in final_df.iterrows():
    form_name_list.append(row['Form Name'])
    

In [79]:
nest_asyncio.apply()

form_embeddings =  asyncio.run(create_embeddings_new(proj, form_name_list, proj.get_variables()['local'].get('default_embeddings_model_id')))




In [80]:
generic_mapping = []
for (index,row) , form_emb in zip(final_df.iterrows()
                                                    ,form_embeddings['response'],
                                                    ):
    generic_mapping.append(
        {
            "ecs_id": row['row_id'],
            "form_id" : row['form_name_id'],
            "validation_id": row['Check ID'],
            "form_name": row['Form Name'],
            "form_domain_name":row['Form Export Name'],
            
            "variable_name":row['Variable'],
            "validation_logic":row['Programming logic\n(this column should be hidden prior to sending to team review)'],
            
            "action": row['Designation'],
            "action_details": row['Query text'],
            "source": "Historic",
            "path":"/Histoical/MAC186_X11-201-00001_Data Validation Plan_V6.0 15Aug2025.xlsx",
            "form_name_vector":form_emb,
           
            
        }
    )

In [81]:
final_df.columns
len(generic_mapping)

637

In [82]:
import json
from opensearchpy.helpers import bulk
from opensearchpy import OpenSearch
opensearch_client = OpensearchUtil(client, proj)

def bulk_insert( index_name, documents):
    project = proj
        
    # Retrieve credentials from Opensearch connection 
    project_configs = project.get_variables()["local"]
    opensearch_connection = project_configs["opensearch_connection"]



    conn_info = client.get_connection(opensearch_connection).get_info()

    opensearchclient = OpenSearch(
            hosts=[{"host": conn_info["params"]["host"], "port": conn_info["params"]["port"]}],
            http_auth = (conn_info["params"]["username"], conn_info["params"]["password"]),
            use_ssl=conn_info["params"]["ssl"],
            verify_certs=False
    )
    

    docs_to_store = []
    for document in documents:

        action = {
            "_op_type": "index",  # Operation type (index = insert)
            "_index": index_name,  # Index name
            "_id": document['ecs_id'],  # Use the unique document ID
            "_source": document  # Document body
        }
        docs_to_store.append(action)

    success, failed = bulk(opensearchclient, docs_to_store)
    print(f"Successfullly indexed {success} documents.")
    print(f"Failed to index {failed} documents.")
    if failed:
        raise Exception("Document indexing encountered an unknown error. ")


# index_name = proj.get_variables()['local'].get('ecs_opensearch')
index_name = "${projectKey}_ecs_index"
dataiku_project_var = "${projectKey}"
if dataiku_project_var in index_name:
    index_name = index_name.replace(dataiku_project_var, opensearch_client.project.project_key).lower()
    print("index_name",index_name)
bulk_insert(index_name, generic_mapping)

{'type': 'ElasticSearch', 'params': {'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'username': 'genai-admin', 'password': 'qPk7Jf5vcyXDS!**332gXTvSfmcauvr9', 'port': 443, 'ssl': True, 'trustAnySSLCertificate': True, 'dialect': 'ES_7', 'dkuProperties': [], 'namingRule': {'indexNameDatasetNamePrefix': '${projectKey}_'}, 'authType': 'PASSWORD', 'oauth': {'refreshTokenRotation': False}, 'aws': {'service': 'OPENSEARCH_SERVERLESS', 'credentialsMode': 'KEYPAIR', 'customAWSCredentialsProviderParams': []}}, 'credentialsMode': 'GLOBAL', 'proxySettingsAsString': ''}
opensearch <OpenSearch([{'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'port': 443}])>
index_name ecsgeneration_ecs_index


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/opensearchpy/connection/http_urllib3.py:214: UserWarning: Connecting to https://aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com:443 using SSL with verify_certs=False is insecure.
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


Successfullly indexed 637 documents.
Failed to index [] documents.


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
